# 🔧 Orqest Tutorial 2: Agents as Tools

Welcome to **Agent as Tools** - one of Orqest's most powerful patterns! This tutorial shows how agents can use other agents as tools.

---

## 🎯 What You'll Learn

- How to use one agent as a tool for another agent
- Why specialized agents are better than one big agent
- How to maintain conversation context between agents

---

## 🌟 The Core Idea: Specialized Agents

Instead of one agent trying to do everything:

| ❌ Traditional | ✅ Orqest Way |
|---------------|---------------|
| One big agent doing planning, chatting, analyzing | Specialized agents: one for planning, one for orchestrating |
| Complex prompts | Simple, focused prompts |
| Hard to debug | Easy to trace which agent did what |

Let's build this! 👇


---

## 📦 Step 1: Imports

Same as Tutorial 1, but we add `RunContext` for tools:


In [72]:
from pydantic import BaseModel, Field
from typing import List, Dict, Any
from pydantic_ai import RunContext

from orqest.agents.base_agent import BaseAgent, NoValidResponse


---

## 🧠 Step 2: Define State and Output Models

Following Orqest's pattern from Tutorial 1, we separate:
- **Global State**: The conversation memory (SimpleState)
- **Output Formats**: Structured response formats (PlanOutput, TextOutput)

This separation gives us the same benefits as Tutorial 1:
- ✅ **Clear Contracts**: Each agent's output format is explicit
- ✅ **Type Safety**: IDEs can autocomplete and catch errors
- ✅ **Easy Integration**: Other components know exactly what to expect
- ✅ **Validation**: Pydantic ensures outputs meet requirements


In [73]:
class SimpleState(BaseModel):
    """Simple state with just messages and a plan."""

    messages: List[Dict[str, str]] = Field(default_factory=list)
    plan: List[str] = Field(default_factory=list)

    def add_message(self, role: str, content: str) -> None:
        """Add a message to the conversation."""
        self.messages.append({"role": role, "content": content})

    def get_user_message(self) -> str:
        """Get the latest user message."""
        for msg in reversed(self.messages):
            if msg["role"] == "user":
                return msg["content"]
        return "No user message"

class PlanOutput(BaseModel):
    """Structured output format for planning responses.

    This defines what the planner agent returns to pydantic-ai,
    separate from the conversation state.
    """
    plan_text: str = Field(
        description="The planning response text",
        min_length=1
    )
    steps_identified: int = Field(
        description="Number of plan steps identified",
        ge=0
    )

class TextOutput(BaseModel):
    """Simple text output format for general responses.

    Used by the orchestrator for non-planning responses.
    """
    answer: str = Field(
        description="The assistant's reply text",
        min_length=1
    )


---

## 🎯 Step 3: Planning Agent (The Specialist)

This agent does ONE thing: creates plans. That's it!


In [74]:
class PlannerAgent(BaseAgent[SimpleState]):
    """A simple planning agent that creates step-by-step plans."""

    def __init__(self):
        super().__init__(
            agent_name="planner",
            output_type=PlanOutput,  # Uses PlanOutput for pydantic-ai
            system_prompt="You are a planning agent. Break down the user's request into 3-5 clear, numbered steps. Be practical and specific.",
            retries=2,
            deps_type=SimpleState
        )

    async def _run_implementation(self, state: SimpleState, **kwargs) -> SimpleState:
        """Create a plan based on the user's request."""
        user_request = state.get_user_message()

        prompt = f"Create a step-by-step plan for: {user_request}"

        response = await self.agent.run(prompt, deps=state, **kwargs)

        return await self._process_response_implementation(response, state, **kwargs)

    async def _process_response_implementation(self, response, state: SimpleState, **kwargs) -> SimpleState:
        """Extract the plan from the response."""
        # Extract content from PlanOutput response
        if hasattr(response, "output") and hasattr(response.output, "plan_text"):
            content = response.output.plan_text
        elif hasattr(response, "content"):
            content = response.content
        else:
            content = str(response)

        # Add response to messages
        state.add_message("assistant", content)

        # Extract numbered steps (simple parsing)
        state.plan.clear()
        lines = content.split('\n')
        for line in lines:
            line = line.strip()
            if line and (line[0].isdigit() or line.startswith(("Step", "•", "-"))):
                # Clean up the step
                clean_step = line.split('.', 1)[-1].strip()
                if clean_step:
                    state.plan.append(clean_step)

        return state


---

## 🎭 Step 4: Orchestrator Agent (Uses Other Agents as Tools)

This is where the magic happens! The orchestrator uses the planner as a **tool**.


In [75]:
class OrchestratorAgent(BaseAgent[SimpleState]):
    """Orchestrator that uses other agents as tools."""

    def __init__(self):
        # Set up tools - including other agents!
        tools = [self._use_planner]

        super().__init__(
            agent_name="orchestrator",
            output_type=TextOutput,  # Uses TextOutput for pydantic-ai
            system_prompt="""You are a helpful orchestrator. When users ask for help planning something, use the 'use_planner' tool. For other questions, respond directly.""",
            retries=2,
            deps_type=SimpleState,
            tools=tools
        )

        # Create the planner agent
        self.planner = PlannerAgent()

    async def _run_implementation(self, state: SimpleState, **kwargs) -> SimpleState:
        """Analyze the request and decide whether to use tools."""
        user_message = state.get_user_message()

        prompt = f"""
        User said: {user_message}

        If this looks like a planning request (organizing, creating steps, etc.), use the use_planner tool.
        Otherwise, respond directly.
        """

        response = await self.agent.run(prompt, deps=state, **kwargs)

        return await self._process_response_implementation(response, state, **kwargs)

    async def _process_response_implementation(self, response, state: SimpleState, **kwargs) -> SimpleState:
        """Process the orchestrator's response."""
        # Extract content from TextOutput response
        if hasattr(response, "output") and hasattr(response.output, "answer"):
            content = response.output.answer
        elif hasattr(response, "content"):
            content = response.content
        else:
            content = str(response)

        state.add_message("assistant", content)
        return state

    # === THE MAGIC: AGENT AS TOOL ===

    async def _use_planner(self, ctx: RunContext[SimpleState], task: str) -> Dict[str, Any]:
        """🔧 Tool: Use the planner agent to create a plan.

        This is the heart of 'Agent as Tools'!
        """
        try:
            # Get current state
            state = ctx.deps

            # Add the planning task as a user message
            state.add_message("user", task)

            # 🔥 Call the planner agent!
            result = await self.planner.run(state)

            if isinstance(result, NoValidResponse):
                return {"success": False, "error": "Planning failed"}

            return {
                "success": True,
                "plan_steps": len(result.plan),
                "plan": result.plan
            }

        except Exception as e:
            return {"success": False, "error": str(e)}


---

## 🎬 Step 5: See It Work!

Let's test our agent-as-tools system:


In [76]:
# Create initial state
state = SimpleState()
state.add_message("user", "Help me plan a birthday party for my friend")

# Create orchestrator (which has planner as a tool)
orchestrator = OrchestratorAgent()

print("🎭 Running orchestrator...")
result = await orchestrator.run(state)

print("\n✨ Done! Let's see what happened:\n")
print(f"📝 Messages: {len(result.messages)}")
print(f"📋 Plan steps: {len(result.plan)}")

# Show the conversation
for i, msg in enumerate(result.messages):
    role = msg["role"].upper()
    content = msg["content"][:100] + "..." if len(msg["content"]) > 100 else msg["content"]
    print(f"\n{i+1}. [{role}]: {content}")

# Show the plan
if result.plan:
    print(f"\n📋 Generated Plan:")
    for i, step in enumerate(result.plan, 1):
        print(f"  {i}. {step}")


🎭 Running orchestrator...


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



✨ Done! Let's see what happened:

📝 Messages: 4
📋 Plan steps: 8

1. [USER]: Help me plan a birthday party for my friend

2. [USER]: Plan a birthday party for a friend

3. [ASSISTANT]: 1. Determine the Date and Time:
   - Discuss with your friend to select a convenient date and time f...

4. [ASSISTANT]: Here's a simple birthday party plan you can follow:

1. **Determine the Date and Time:** Decide when...

📋 Generated Plan:
  1. Determine the Date and Time:
  2. Choose a Venue:
  3. Create a Guest List:
  4. Send Invitations:
  5. Plan the Menu and Activities:
  6. Decorate and Prepare:
  7. Plan for a Cake and Gifts:
  8. Confirm Attendance and Finalize Details:


---

## 🔄 Step 6: Continue the Conversation

The beauty of this approach: the state carries forward naturally!


In [77]:
# Add a follow-up question
result.add_message("user", "Make it more budget-friendly")

print("🎭 Running with follow-up...")
updated_result = await orchestrator.run(result)

print("\n✨ Updated response:")
latest_msg = updated_result.messages[-1]["content"]
print(f"🤖 {latest_msg[:200]}...")

print(f"\n📊 Total messages now: {len(updated_result.messages)}")
print(f"📋 Plan steps now: {len(updated_result.plan)}")


🎭 Running with follow-up...


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



✨ Updated response:
🤖 To make your plan more budget-friendly, follow these steps:

1. **Review the Current Plan**: Thoroughly analyze the existing plan to understand all its components, costs, and specifics. Make a detaile...

📊 Total messages now: 8
📋 Plan steps now: 5


---

## 🎯 Step 7: Test Different Request Types

Let's see how the orchestrator handles different types of requests:


In [61]:
# Test 1: Non-planning request
simple_state = SimpleState()
simple_state.add_message("user", "What's the weather like?")

simple_result = await orchestrator.run(simple_state)
print("🧪 Test 1 - Weather question:")
print(f"   Response: {simple_result.messages[-1]['content'][:100]}...")
print(f"   Plan created: {'Yes' if simple_result.plan else 'No'}")

# Test 2: Planning request
planning_state = SimpleState()
planning_state.add_message("user", "Help me plan a study schedule for learning Python")

planning_result = await orchestrator.run(planning_state)
print(f"\n🧪 Test 2 - Study planning:")
print(f"   Response: {planning_result.messages[-1]['content'][:100]}...")
print(f"   Plan steps: {len(planning_result.plan)}")
if planning_result.plan:
    print(f"   First step: {planning_result.plan[0]}")


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


🧪 Test 1 - Weather question:
   Response: I can't provide the current weather information. Please check a reliable weather website or app for ...
   Plan created: No


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



🧪 Test 2 - Study planning:
   Response: Here's a plan to help you learn Python effectively:

1. **Identify Learning Goals:** Determine what ...
   Plan steps: 10
   First step: **Identify Learning Goals:**


---

## 🎓 What You've Built

Congratulations! You've implemented the **Agent as Tools** pattern with:

### 🏗️ **Simple Architecture**
```
🎭 OrchestratorAgent (Coordinator)
└── 🎯 PlannerAgent (Tool/Specialist)
```

### 🌟 **Key Benefits You've Seen**

| Benefit | What You Built |
|---------|----------------|
| **Specialization** | Planner only does planning, orchestrator only coordinates |
| **Reusability** | Planner can be used by other agents too |
| **Clarity** | Easy to see which agent did what |
| **Testability** | Can test each agent independently |

### 🚀 **Why This Scales**

This simple pattern lets you:
- Add more specialist agents (executor, analyzer, etc.)
- Chain agents together for complex workflows
- Mix and match agents for different use cases
- Keep each agent simple and focused

### 🎯 **Next Steps**

Try adding:
1. **ExecutorAgent**: Takes plans and executes them
2. **AnalyzerAgent**: Reviews results and provides insights
3. **More tools**: Web search, database queries, etc.

The pattern stays the same - each agent does one thing well, and agents use each other as tools! 🚀

---

## 🎉 You Did It!

You've mastered the **Agent as Tools** pattern - one of the most powerful concepts in modular AI development.

**The key insight**: Instead of building one complex agent, build simple agents that work together. This makes everything easier to understand, test, and extend!

Welcome to truly modular AI development! 🌟
